# 01 - Exploratory Data Analysis

Explore the bundled synthetic support knowledge base and evaluation tickets that power every other notebook and the `src/` pipeline.

In production this would run against the real 500k-ticket corpus and the full technical documentation set; the same code works unchanged.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src.data_engineering.ingestion import load_sample_knowledge_base

## Knowledge base overview

In [2]:
documents = load_sample_knowledge_base(Path.cwd().parent / 'data/sample/knowledge_base.json')
kb_df = pd.DataFrame([
    {'doc_id': d.doc_id, 'title': d.title, 'n_words': len(d.text.split()), 'page': d.metadata.get('page')}
    for d in documents
])
kb_df

,doc_id,title,n_words,page
0,billing_policy.pdf,Billing Policy,44,12
1,duplicate_payment_procedure.pdf,Duplicate Payment Procedure,42,3
2,refund_policy.pdf,Refund Policy,33,1
3,subscription_cancellation.pdf,Subscription Cancellation,31,5
4,account_creation.pdf,Account Creation,28,1
5,payment_troubleshooting.pdf,Payment Troubleshooting,37,8
6,password_reset.pdf,Password Reset,28,2
7,premium_features.pdf,Premium Features,22,4
8,data_export_guide.pdf,Data Export Guide,29,6
9,two_factor_auth.pdf,Two-Factor Authentication,28,2


In [3]:
kb_df['n_words'].describe()

count    12.000000
mean     32.500000
std       6.501748
min      22.000000
25%      28.000000
50%      30.500000
75%      37.250000
max      44.000000
Name: n_words, dtype: float64

## Evaluation tickets overview

In [4]:
tickets = json.loads((Path.cwd().parent / 'data/sample/eval_tickets.json').read_text())
tickets_df = pd.DataFrame(tickets)
tickets_df

,query,intent,priority,sentiment,relevant_docs,escalation_required
0,I was charged twice for my Premium subscriptio...,billing_issue,high,negative,"[billing_policy.pdf, duplicate_payment_procedu...",True
1,How do I request a refund for my annual plan?,refund_request,medium,neutral,[refund_policy.pdf],False
2,How can I cancel my subscription before the re...,cancellation,low,neutral,[subscription_cancellation.pdf],False
3,"My credit card payment keeps failing, what sho...",payment_issue,medium,negative,[payment_troubleshooting.pdf],False
4,I forgot my password and the reset email never...,account_access,medium,negative,[password_reset.pdf],False
5,What is included in the Premium plan compared ...,product_question,low,neutral,[premium_features.pdf],False
6,How do I export all of my account data?,data_export,low,neutral,[data_export_guide.pdf],False
7,Can I enable two factor authentication with an...,security_question,low,neutral,[two_factor_auth.pdf],False
8,"I keep hitting rate limit errors on the API, h...",api_question,medium,negative,"[api_rate_limits.pdf, premium_features.pdf]",False
9,This is the third time I've been billed incorr...,billing_issue,high,negative,"[billing_policy.pdf, escalation_policy.pdf]",True


In [5]:
tickets_df['intent'].value_counts()

intent
billing_issue        2
refund_request       1
cancellation         1
payment_issue        1
account_access       1
product_question     1
data_export          1
security_question    1
api_question         1
Name: count, dtype: int64

In [6]:
tickets_df['priority'].value_counts()

priority
medium    4
low       4
high      2
Name: count, dtype: int64

## Chunking strategy comparison

Compare the number of chunks produced by each strategy on the same document.

In [7]:
from src.data_engineering.chunking import (
    fixed_size_chunking, overlapping_chunking, recursive_chunking,
)

doc = documents[1]  # duplicate_payment_procedure.pdf
strategies = {
    'fixed-size (64 tok)': fixed_size_chunking(doc, chunk_size=64),
    'overlapping (64/16)': overlapping_chunking(doc, chunk_size=64, overlap=16),
    'recursive': recursive_chunking(doc, chunk_size=64),
}
pd.DataFrame([
    {'strategy': name, 'n_chunks': len(chunks), 'avg_chunk_words': sum(len(c.text.split()) for c in chunks) / len(chunks)}
    for name, chunks in strategies.items()
])

,strategy,n_chunks,avg_chunk_words
0,fixed-size (64 tok),1,42.0
1,overlapping (64/16),1,42.0
2,recursive,1,42.0


## Takeaways

- The sample KB has 12 short policy documents; the real corpus would have thousands of pages, making chunking strategy choice far more impactful on retrieval quality.
- `billing_issue` tickets are the highest-priority, highest-escalation-rate category in the sample set, which is the scenario the RAG + reranker + fine-tuned LLM stack is optimized for.
- Recursive chunking preserves paragraph structure and produces fewer, more semantically coherent chunks than fixed-size splitting on these short documents.